#### Metrics to evaluate perception 

- **Consistency in detection**: How often does the perception model detect the object when it is present?

- **Consistency in identification**: How often does the perception model correctly identify the object when it is detected?

- **Detection range**: How far away can the perception model detect the object?

- **Detection latency**: How long does it take for the perception model to detect the object after it enters the sensor's field of view?

- **Positional accuracy**: How accurate is the perceived position of the object compared to its true position?

- **Dimensional accuracy**: How accurate are the perceived dimensions of the object compared to its true dimensions?


Setup directories and import libraries

In [1]:
from pathlib import Path
import subprocess
import sys
import tkinter as tk
from tkinter import filedialog

project_root = Path.cwd().parent
script = project_root / "scripts" / "run_perception.py"
results = project_root / "results"
results.mkdir(exist_ok=True)

Sensor configuration parameters: add/change for sensor experiments.

In [51]:
sensor_configs = [
    # {
    #     "name": "short_narrow",
    #     "far_range": "50",
    #     "fov": "50",
    # },
    
    # {
    #     "name": "medium_wide",
    #     "far_range": "100",
    #     "fov": "70",
    # },
    
    {
        "name": "long_wide",
        "far_range": "500",
        "fov": "120",
    },
]

fixed_sensor_args = [
    "--ego-id", "0",
    "--sensor-x", "4.0",
    "--sensor-y", "0.0",
    "--sensor-z", "0.5",
    "--sensor-heading", "0.0",
    "--near-range", "1.0",
    "--max-detections", "100",
]


runtime_args = [
    "--dt", "0.05",
    "--realtime-factor", "1", # 0 to run as fast as possible, 1 to run in real time
    "--viewer",
    "--no-keep-viewer-open",
]

MAX_RUNTIME_SECONDS = 3600

Select NCAP scenario

In [42]:
import tkinter as tk
from tkinter import filedialog

scenario_root = project_root / "OSC-NCAP-scenarios" / "OpenSCENARIO" / "NCAP"
root = tk.Tk()
root.withdraw()

# to select file file using method askopenfilename() 
selected_scenario = filedialog.askopenfilename(
    title="Select OpenSCENARIO file",
    initialdir=str(scenario_root),
    filetypes=[("OpenSCENARIO files", "*.xosc"), ("All files", "*.*")],
)
root.destroy()

SCENARIO = Path(selected_scenario)

Batch run for each sensor config, each will generate a csv file, both for the noise-free and noisy perception models. 

CSV files are saved in `results/`

In [52]:
print(f"Selected scenario: {SCENARIO}")

for config in sensor_configs:
    for noise_flag, noise_name in [
        ("--no-perception-noise", "noise_free"),
        ("--perception-noise", "noisy"),
    ]:
        output_file = results / f"{config['name']}_{noise_name}.csv"

        command = [
            sys.executable,
            str(script),
            "--scenario", str(SCENARIO),
            *fixed_sensor_args,
            "--far-range", config["far_range"],
            "--fov-deg", config["fov"],
            *runtime_args,
            noise_flag,
            "--output", str(output_file),
        ]

        print(f"Running {SCENARIO.name} / {config['name']} / {noise_name}...", flush=True)
        subprocess.run(command, check=True, timeout=MAX_RUNTIME_SECONDS)

print("All batch simulations done")

Selected scenario: /home/danilo/Desktop/perception-models/OSC-NCAP-scenarios/OpenSCENARIO/NCAP/AEB_VRU_2023/NCAP_AEB_VRU_CBFA_2023.xosc
Running NCAP_AEB_VRU_CBFA_2023.xosc / long_wide / noise_free...
Saving results to /home/danilo/Desktop/perception-models/results/long_wide_noise_free.csv
Running NCAP_AEB_VRU_CBFA_2023.xosc / long_wide / noisy...
Saving results to /home/danilo/Desktop/perception-models/results/long_wide_noisy.csv
All batch simulations done


#### Structure of csv file

In [ ]:
%matplotlib widget
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



Starting batch 
